<a href="https://colab.research.google.com/github/charan0318/InSAR-KG-Basin-Subsidence/blob/main/InSAR_KG_Basin_Subsidence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time-Series InSAR Analysis using MintPy
## Study Area: Krishna–Godavari Basin Subsidence

This notebook demonstrates a complete multi-temporal InSAR workflow using **MintPy** to process Sentinel-1 interferograms generated via the ASF HyP3 processor. The goal is to estimate ground deformation velocity and identify subsidence patterns in the Krishna-Godavari basin.

In [ ]:
!pip install mintpy --upgrade

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from google.colab import drive
from IPython.display import Image, display

# Mount Google Drive
drive.mount('/content/drive')

# Set Environment Variables for performance stability
os.environ["OMP_NUM_THREADS"] = "1"

# Define Project Paths
base_dir = "/content/drive/MyDrive/InSAR_Project"
work_dir = os.path.join(base_dir, "mintpy")
data_dir = os.path.join(base_dir, "hyp3_extracted")

if not os.path.exists(work_dir):
    os.makedirs(work_dir)

os.chdir(work_dir)
print(f"Working directory set to: {os.getcwd()}")

## Data Description

This analysis utilizes the following dataset:
* **Sensor**: Sentinel-1 (C-band)
* **Product Type**: SLC (Level-1) processed to Interferograms
* **Mode**: Interferometric Wide (IW)
* **Polarization**: VV+VH
* **Orbit**: Descending (Path 19)
* **Timeframe**: 2018–2022
* **Quantity**: ~111 interferograms derived from ~55 acquisition dates.

In [ ]:
# Verify input data structure
folders = sorted(os.listdir(data_dir))
print(f"Total interferogram folders found: {len(folders)}")

if len(folders) > 0:
    sample_folder = os.path.join(data_dir, folders[0])
    print(f"Sample folder: {folders[0]}")
    print("Files inside sample folder:")
    display(os.listdir(sample_folder)[:5])

## MintPy Configuration

We define a configuration template that tells MintPy where to find the HyP3 GeoTIFF files and which processing parameters to use. We explicitly disable atmospheric and LOD corrections to ensure stability in the cloud environment.

In [ ]:
config_file = 'smallbaselineApp.cfg'

# Generate default config and append specific project parameters
!smallbaselineApp.py -H > {config_file}

with open(config_file, "a") as f:
    f.write(f"""
########## 1. load_data
mintpy.load.processor = hyp3
mintpy.load.unwFile = {data_dir}/*/*unw*.tif
mintpy.load.corFile = {data_dir}/*/*corr.tif
mintpy.load.lookupYFile = {data_dir}/*/*lv_theta*.tif
mintpy.load.lookupXFile = {data_dir}/*/*lv_phi*.tif
mintpy.load.demFile = {data_dir}/*/*dem.tif
mintpy.load.incAngleFile = {data_dir}/*/*inc_map*.tif

########## 2. processing parameters
mintpy.troposphericDelay.method = no
mintpy.correct.LOD = no
mintpy.topographicResidual = no
mintpy.compute.cluster = local
mintpy.compute.numWorker = 4
""")

print("Configuration file smallbaselineApp.cfg generated successfully.")

## MintPy Processing Pipeline

The following steps execute the core InSAR time-series logic: loading the data into HDF5 stacks, inverting the network to find displacements, and estimating the linear velocity.

**Note**: The Small Baseline Subset (SBAS) approach implemented in MintPy was used to minimize temporal and spatial decorrelation effects and improve the robustness of time-series inversion.

In [ ]:
# Step 1: Load data
!smallbaselineApp.py {config_file} --dostep load_data

# Step 2: Network Inversion
# Remove previous HDF5 outputs to ensure a clean processing run if interrupted
!rm -f timeseries.h5 temporalCoherence.h5 numInvIfgram.h5
!smallbaselineApp.py {config_file} --start invert_network --end velocity

## Velocity Output

The estimated LOS (Line-of-Sight) velocity is displayed below.

**Interpretation Note**: The LOS velocity is derived from MintPy and typically expressed in meters per year (m/year). For visualization here, `view.py` provides a spatial overview of deformation patterns relative to the reference pixel.

In [ ]:
output_image = 'velocity_map.png'
!view.py velocity.h5 velocity -o {output_image}
display(Image(filename=output_image))

## Export to GeoTIFF

To facilitate GIS analysis, we export the HDF5 velocity data to a GeoTIFF format and scale the values to **mm/year**.

**Quality Control**: A temporal coherence mask (`maskTempCoh.h5`) is applied during export to suppress low-quality pixels and focus on areas with stable phase signals.

In [ ]:
# Save masked velocity as GeoTIFF
!save_gdal.py velocity.h5 -m maskTempCoh.h5 -o velocity_masked.tif

# Convert units from m/year to mm/year using rasterio
with rasterio.open("velocity_masked.tif") as src:
    data = src.read(1)
    profile = src.profile

data_mm = data * 1000

with rasterio.open("velocity_masked_mm.tif", "w", **profile) as dst:
    dst.write(data_mm.astype(np.float32), 1)

print(f"✅ Exported velocity_masked_mm.tif | Min: {np.nanmin(data_mm):.2f} mm/y, Max: {np.nanmax(data_mm):.2f} mm/y")

## Time-Series Visualization

We extract the displacement history for a specific pixel of interest to observe the temporal evolution of deformation.

In [ ]:
with h5py.File("timeseries.h5", "r") as f:
    ts = f["timeseries"][:]
    dates = [d.decode('utf-8') for d in f["date"][:]]

y, x = 1500, 1800  # Example coordinates
window_size = 5
pixel_ts = ts[:, y, x] * 1000 # Convert to mm
mean_ts = np.nanmean(ts[:, y-2:y+3, x-2:x+3], axis=(1,2)) * 1000

plt.figure(figsize=(12, 5))
plt.plot(dates, pixel_ts, label='Single Pixel', alpha=0.5)
plt.plot(dates, mean_ts, marker='o', label='5x5 Area Average', linewidth=2)
plt.xticks(rotation=45)
plt.ylabel("Displacement (mm)")
plt.title("Deformation Time-Series")
plt.legend()
plt.grid(True)
plt.show()

## Important Scientific Notes

* **Relative Deformation**: All measurements are relative to the reference pixel and the first acquisition date.
* **LOS Geometry**: The values represent movement in the Line-of-Sight (LOS) toward or away from the satellite, not purely vertical motion.
* **Sign Convention**: Negative values (blue/cool colors) indicate movement **away** from the satellite (subsidence), while positive values indicate uplift.
* **Corrections**: Atmospheric corrections (ERA5/GACOS) were omitted in this run to prioritize baseline processing stability.

## Output Summary

The resulting `velocity_masked_mm.tif` is now ready for spatial analysis in GIS software (e.g., QGIS).

**Suggested GIS Workflow:**
1. Filter values between -20 and 0 mm/year to focus on subsidence.
2. Apply a threshold (< -10 mm/year) to isolate high-risk zones.
3. Convert the raster to polygons for intersection with infrastructure or land-use maps.